In [1]:
import sys
from pathlib import Path
import os
import laspy
import numpy as np
import pandas as pd
import time 

In [9]:
from GlobalEvaluator2 import GlobalEvaluator, compute_tile_metrics
root = Path("../..").resolve()
sys.path.append(str(root))

from scripts.treeSegWatershed import classify_tree_watershed


In [19]:
import pandas as pd
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_watershed_trees(df):
    """
    Visualizes a point cloud where each 'label' has a distinct color.
    """
    # 1. Extract coordinates and normalize (important for UTM coordinates)
    points = df[['X', 'Y', 'Z']].values
    center = points.mean(axis=0)
    points_normalized = points - center

    # 2. Create the Open3D point cloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_normalized)

    # 3. Assign colors based on the label
    labels = df['label'].values
    unique_labels = np.unique(labels)
    
    # Use a matplotlib colormap for visually distinct colors
    cmap = plt.get_cmap("tab20")  # 'tab20' provides 20 well-differentiated colors
    
    colors = np.zeros((len(labels), 3))
    
    for i, label in enumerate(unique_labels):
        # Label -1 (usually ground/noise in Watershed) is rendered as grey
        if label == -1:  # -1 is treated as background
            colors[labels == label] = [0.2, 0.2, 0.2]
        else:
            # Assign color by unique-label index
            colors[labels == label] = cmap(i % 20)[:3]

    pcd.colors = o3d.utility.Vector3dVector(colors)

    # 4. Configure the visualizer
    print(f"Visualizing {len(unique_labels)} unique labels.")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Watershed Results - Santomera", width=1024, height=768)
    
    vis.add_geometry(pcd)
    
    # Improve visualization (dark background and point size)
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 2.0
    
    vis.run()
    vis.destroy_window()


In [ ]:
def combine_df_las(trees_df, tree_las_gt):
    # Read the ground-truth LAS file
    gt_las_points = laspy.read(tree_las_gt)

    # Build a temporary DataFrame with GT coordinates
    # Extract X, Y, Z from the LAS for comparison
    gt_df = pd.DataFrame({
        'X': np.array(gt_las_points.x),
        'Y': np.array(gt_las_points.y),
        'Z': np.array(gt_las_points.z)
    })


    trees_df[['X', 'Y', 'Z']] = trees_df[['X', 'Y', 'Z']]
    gt_df[['X', 'Y', 'Z']] = gt_df[['X', 'Y', 'Z']]

    
    trees_df = trees_df.drop_duplicates(subset=['X', 'Y', 'Z'])
    # Filter via a left merge
    # Only rows from trees_df whose coordinates exist in gt_df are kept
    trees_df = pd.merge(gt_df, trees_df, on=['X', 'Y', 'Z'], how='left')

    # Points the watershed did NOT find appear as NaN.
    # Filling with -1 marks them as unpredicted (no tree found at that point).
    trees_df['label'] = trees_df['label'].fillna(-1)

    return trees_df

In [4]:
def generate_empty_result(trees_path):
    gt_las_points = laspy.read(trees_path)
    gt_df = pd.DataFrame({
        'X': np.array(gt_las_points.x),
        'Y': np.array(gt_las_points.y),
        'Z': np.array(gt_las_points.z)
    })
    gt_df['label'] = -1  # Assign -1 to all points to indicate "not found"
    return gt_df

In [ ]:
def process_and_test_watershed(trees_path, terrain_path, resolution, min_tree_height, sphericity_threshold, evaluator=None):
 
    # Apply watershed to segment trees
    trees_df = classify_tree_watershed(trees_path, terrain_path, resolution, min_tree_height, sphericity_threshold)
    
    #print(f"visualize_watershed_trees for {trees_path}")
    if trees_df.empty:
        trees_df = generate_empty_result(trees_path)
        print("No trees found with the given parameters for {0}.".format(trees_path))
        return
    # visualize_watershed_trees(trees_df)

    trees_df_copy = trees_df.copy()

    trees_df = combine_df_las(trees_df, trees_path)

    

    
    # # Compute evaluation metrics
    tile_metrics = compute_tile_metrics(trees_df, trees_path)
    evaluator.accumulate_tile(tile_metrics)

    return trees_df_copy


In [ ]:
def process_folder_watershed(trees_folder_path, terrain_folder_path, resolution, min_tree_height, sphericity_threshold):
    evaluator = GlobalEvaluator()
    initial_time = time.time()

    
    for filename in os.listdir(trees_folder_path):
        if filename.endswith(".las"):
            trees_path = os.path.join(trees_folder_path, filename)
            terrain_path = os.path.join(terrain_folder_path, filename)
            process_and_test_watershed(trees_path, terrain_path, resolution,
            min_tree_height,
            sphericity_threshold, evaluator=evaluator)

    print(f"Number of predicted trees: {len(evaluator.all_pred_instances)}")
    end_time = time.time()

    return evaluator.report_micro_global(), (end_time - initial_time)/60

## TEST

In [ ]:
trees_folder_path = "../../data/Santomera/tests/"
terrain_folder_path = "../../data/Santomera/tiles/terrain/"
report, elapsed_time = process_folder_watershed(trees_folder_path, terrain_folder_path, 0.5, 1.2, 0.05)
print(report)
print(f"Total processing time: {elapsed_time:.2f} minutes")

Visualizando 10 etiquetas únicas.
Visualizando 9 etiquetas únicas.
Visualizando 8 etiquetas únicas.
Visualizando 10 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 4 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 5 etiquetas únicas.
Numero de arboles predichos: 64
   Categoría              Métrica     Valor Valor (%)
0  Semántica        IoU Semántico  0.721883    72.19%
1  Semántica   F1-Score Semántico  0.838481    83.85%
2  Instancia  Precision (Inst@50)  0.484375    48.44%
3  Instancia     Recall (Inst@50)  0.260504    26.05%
4  Instancia   F1-Score (Inst@50)  0.338798    33.88%
5  Instancia                mWCov  0.470268    47.03%
6  Instancia                 AP50  0.122258    12.23%
7  Instancia                 AP25  0.427757    42.78%
Tiempo total de procesamiento: 0.46 minutos
